In [ ]:
repo_root = Path('/playpen-ssd/wokwen/projects/autoeval_chatbot')

# Load data_no_prompt
data_no_prompt_criteria = pd.read_csv(repo_root / 'ratings' / 'qwen' / 'baseline' / 'data_no_prompt' / 'criteria_ratings.csv')
data_no_prompt_overall = pd.read_csv(repo_root / 'ratings' / 'qwen' / 'baseline' / 'data_no_prompt' / 'overall_ratings.csv')

# Load prompt_no_data
prompt_no_data_criteria = pd.read_csv(repo_root / 'ratings' / 'qwen' / 'baseline' / 'prompt_no_data' / 'criteria_ratings.csv')
prompt_no_data_overall = pd.read_csv(repo_root / 'ratings' / 'qwen' / 'baseline' / 'prompt_no_data' / 'overall_ratings.csv')

print('Data no prompt criteria shape:', data_no_prompt_criteria.shape)
print('Data no prompt overall shape:', data_no_prompt_overall.shape)
print('Prompt no data criteria shape:', prompt_no_data_criteria.shape)
print('Prompt no data overall shape:', prompt_no_data_overall.shape)

In [ ]:
# Merge criteria and overall for each baseline type
data_no_prompt = data_no_prompt_criteria.merge(data_no_prompt_overall, on='Conversation_Id')
prompt_no_data = prompt_no_data_criteria.merge(prompt_no_data_overall, on='Conversation_Id')

# Add baseline type column
data_no_prompt['baseline_type'] = 'data_no_prompt'
prompt_no_data['baseline_type'] = 'prompt_no_data'

# Combine into one dataframe
df = pd.concat([data_no_prompt, prompt_no_data], ignore_index=True)
print('Combined df shape:', df.shape)
print('Columns:', df.columns.tolist())

In [ ]:
# Summary statistics for each baseline type
summary = df.groupby('baseline_type').agg({
    col: ['mean', 'std', 'count'] for col in df.columns if col not in ['Conversation_Id', 'baseline_type']
})
display(summary)

In [ ]:
# Visualize distributions of criteria ratings by baseline type
criteria_cols = [c for c in df.columns if '_Rating' in c and 'baseline_type' not in c and 'Conversation_Id' not in c]
fig, axes = plt.subplots(len(criteria_cols), 1, figsize=(10, 4*len(criteria_cols)))
if len(criteria_cols) == 1:
    axes = [axes]
for i, col in enumerate(criteria_cols):
    sns.boxplot(data=df, x='baseline_type', y=col, ax=axes[i])
    axes[i].set_title(f'Distribution of {col} by Baseline Type')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis within each baseline type
for baseline in ['data_no_prompt', 'prompt_no_data']:
    subset = df[df['baseline_type'] == baseline]
    numeric_cols = subset.select_dtypes(include=[np.number]).columns
    corr = subset[numeric_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title(f'Correlation Matrix for {baseline}')
    plt.show()

In [ ]:
# Statistical tests: compare means between baseline types for each criterion
from scipy.stats import ttest_ind
results = []
for col in criteria_cols + ['User_Rating', 'Observer_Rating', 'Self_Rating']:
    group1 = df[df['baseline_type'] == 'data_no_prompt'][col].dropna()
    group2 = df[df['baseline_type'] == 'prompt_no_data'][col].dropna()
    if len(group1) > 1 and len(group2) > 1:
        t_stat, p_val = ttest_ind(group1, group2)
        results.append({
            'criterion': col,
            'mean_data_no_prompt': group1.mean(),
            'mean_prompt_no_data': group2.mean(),
            't_stat': t_stat,
            'p_value': p_val,
            'significant': p_val < 0.05
        })
results_df = pd.DataFrame(results)
display(results_df)